# Xarray-Spatial Erosion: Particle-based hydraulic erosion

The `erode` function simulates hydraulic erosion by dropping thousands of virtual water droplets onto a terrain raster. Each droplet traces a path downhill, picking up sediment on steep slopes and depositing it on flat ground. The result is realistic valley networks and smoothed ridgelines from synthetic or real elevation data.

### What you'll build

1. Generate synthetic terrain from layered sine waves
2. Run basic erosion and compare before/after elevation
3. Explore how erosion, capacity, and brush radius affect channel formation
4. Watch channel networks deepen with increasing droplet counts
5. Compare cross-sections through original and eroded terrain

![Hydraulic erosion preview](images/hydraulic_erosion_preview.png)

**Jump to a section:**
[Basic erosion](#Basic-erosion) | [Parameter effects](#Parameter-effects) | [Iterative refinement](#Iterative-refinement) | [Cross-section](#Cross-section)

Standard imports plus `erode` from xrspatial.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from xrspatial.erosion import erode

## Synthetic terrain

Layered sine waves plus random noise give enough slope variation for erosion to produce visible channels. The 256x256 grid runs quickly while still showing detailed channel networks.

In [ ]:
size = 256
y = np.linspace(0, 4 * np.pi, size)
x = np.linspace(0, 4 * np.pi, size)
xx, yy = np.meshgrid(x, y)

rng = np.random.default_rng(42)
terrain = (
    200 * np.sin(xx * 0.3) * np.cos(yy * 0.2)
    + 100 * np.sin(xx * 0.7 + yy * 0.5)
    + 50 * rng.random((size, size))
    + 500
)

agg = xr.DataArray(
    terrain.astype(np.float32),
    dims=['y', 'x'],
    coords={'y': np.arange(size, dtype=float), 'x': np.arange(size, dtype=float)},
    attrs={'res': (1.0, 1.0)},
)

fig, ax = plt.subplots(figsize=(10, 7.5))
agg.plot.imshow(ax=ax, cmap='terrain', add_colorbar=True,
                cbar_kwargs={'label': 'Elevation'})
ax.set_title('Original terrain')
ax.set_axis_off()
plt.tight_layout()

## Basic erosion

Running `erode` with 50,000 droplets carves channels into the terrain. The difference map (right panel) shows where material was removed (blue) and deposited (orange). Blue channels follow the steepest descent paths.

In [ ]:
eroded = erode(agg, iterations=50000, seed=42)

diff = eroded - agg
vlim = float(max(abs(diff.min()), abs(diff.max())))

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

agg.plot.imshow(ax=axes[0], cmap='terrain', add_colorbar=True,
                cbar_kwargs={'label': 'Elevation'})
axes[0].set_title('Before')
axes[0].set_axis_off()

eroded.plot.imshow(ax=axes[1], cmap='terrain', add_colorbar=True,
                   cbar_kwargs={'label': 'Elevation'})
axes[1].set_title('After erosion (50k droplets)')
axes[1].set_axis_off()

diff.plot.imshow(ax=axes[2], cmap='coolwarm', vmin=-vlim, vmax=vlim,
                 add_colorbar=True, cbar_kwargs={'label': 'Elevation change'})
axes[2].set_title('Difference')
axes[2].set_axis_off()

plt.tight_layout()

# Save preview image
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/hydraulic_erosion_preview.png', bbox_inches='tight', dpi=120)

## Parameter effects

The `params` dict controls erosion behavior. The most important parameters:

- **erosion**: how aggressively particles remove material (default 0.3)
- **capacity**: how much sediment a droplet can carry (default 4.0)
- **deposition**: how quickly excess sediment is dropped (default 0.3)
- **radius**: size of the erosion brush in cells (default 3)

The plots below show elevation change maps for each parameter variation.

In [ ]:
configs = {
    'Default': None,
    'High erosion': {'erosion': 0.9},
    'Large brush (r=6)': {'radius': 6},
    'High capacity': {'capacity': 12.0},
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (label, p) in zip(axes, configs.items()):
    result = erode(agg, iterations=30000, seed=42, params=p)
    d = result - agg
    vlim = float(max(abs(d.min()), abs(d.max()), 1))
    d.plot.imshow(ax=ax, cmap='coolwarm', vmin=-vlim, vmax=vlim,
                  add_colorbar=False)
    ax.set_title(label, fontsize=11)
    ax.set_axis_off()

plt.suptitle('Elevation change under different parameters', y=1.02)
plt.tight_layout()

<div class="alert alert-block alert-warning">
<b>Parameter sensitivity.</b> The <code>erosion</code> and <code>capacity</code> values interact with each other and with the elevation range of your data. Parameters tuned for a 500m relief surface will over-erode a 50m relief surface. Start with defaults and adjust one parameter at a time.
</div>

## Iterative refinement

More iterations means more droplets and deeper, more developed channel networks. At low counts the surface has scattered shallow scratches. At high counts connected valley systems emerge.

In [ ]:
iter_counts = [5000, 20000, 50000, 100000]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, n_iter in zip(axes, iter_counts):
    result = erode(agg, iterations=n_iter, seed=42)
    result.plot.imshow(ax=ax, cmap='terrain', add_colorbar=False)
    ax.set_title(f'{n_iter:,} droplets', fontsize=11)
    ax.set_axis_off()

plt.suptitle('Channel development vs. iteration count', y=1.02)
plt.tight_layout()

## Cross-section

A 1D slice through the terrain shows how erosion carves valleys and smooths peaks. The profile comparison makes it easy to see where material was removed versus redistributed.

In [ ]:
row = size // 2
eroded_heavy = erode(agg, iterations=80000, seed=42)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(agg.values[row, :], label='Original', linewidth=2, color='#1b9e77')
ax.plot(eroded.values[row, :], label='50k droplets', linewidth=1.5, color='#d95f02')
ax.plot(eroded_heavy.values[row, :], label='80k droplets', linewidth=1.5, color='#7570b3')
ax.set_xlabel('Column')
ax.set_ylabel('Elevation')
ax.set_title(f'Cross-section at row {row}')
ax.legend()
plt.tight_layout()

<div class="alert alert-block alert-info">
<b>GPU acceleration.</b> For large grids (1024x1024+), pass a CuPy-backed DataArray to <code>erode</code>. Each droplet runs as a separate CUDA thread, giving significant speedups over the CPU path. The API is identical.
</div>

### References

- [Hydraulic erosion (Wikipedia)](https://en.wikipedia.org/wiki/Hydraulic_action)
- Mei, X., Decaudin, P., & Hu, B. (2007). [Fast Hydraulic Erosion Simulation and Visualization on GPU](https://hal.inria.fr/inria-00402079). *Pacific Graphics*.
- [xrspatial.erosion.erode API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.erosion.erode.html)